In [ ]:
import os
import re
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
from scipy import stats
from pathlib import Path
from mpl_toolkits.mplot3d import Axes3D

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("colorblind")

# Define metric names and order
METRICS = [
    ('share_ratio', 'Share Ratio'),
    ('avg_price', 'Average Price'),
    ('consumer_surplus', 'Average Consumer Surplus'),
    ('firm_surplus', 'Average Firm Surplus'),
    ('avg_search_cost', 'Average Search Cost')
]

def parse_experiment_folder(folder_name: str) -> Dict[str, str]:
    """Parse experiment folder name, extract configuration parameters"""
    result = {
        "enable_cot": False,
        "memory_truncate": 0,
        "memory": False,
        "share_memory": False,
        "share_memory_truncate": 0,
        "rational_share": False,
        "rational_search": False,
        "rational_price": False,
        "model_name": None,
        "original_folder": folder_name
    }

    if folder_name.startswith("cot-memory-0-llm_"):
        result["enable_cot"] = True
        result["memory_truncate"] = 0
        result["memory"] = True
        result["model_name"] = folder_name[len("cot-memory-0-llm_"):]
        return result

    if folder_name.startswith("memory-0-llm_"):
        result["memory_truncate"] = 0
        result["memory"] = True
        result["model_name"] = folder_name[len("memory-0-llm_"):]
        return result

    if folder_name.startswith("memory-0-rational-"):
        result["memory_truncate"] = 0
        result["memory"] = True
        rational_part = folder_name[len("memory-0-rational-"):].split("_")[0]
        if "search+share" in rational_part:
            result["rational_search"] = True
            result["rational_share"] = True
        elif "search" in rational_part:
            result["rational_search"] = True
        elif "share" in rational_part:
            result["rational_share"] = True
        elif "price" in rational_part:
            result["rational_price"] = True
        model_part = folder_name.split("_", 1)[1] if "_" in folder_name else folder_name[len("memory-0-rational-"):]
        result["model_name"] = model_part
        return result

    memory_match = re.match(r"memory-(\d+)-llm_deepseek-v3", folder_name)
    if memory_match:
        result["memory_truncate"] = int(memory_match.group(1))
        result["memory"] = True
        result["model_name"] = "deepseek-v3"
        return result

    share_memory_match = re.match(r"sharememory-(\d+)-llm_grok-3-mini", folder_name)
    if share_memory_match:
        result["share_memory_truncate"] = int(share_memory_match.group(1))
        result["share_memory"] = True
        result["model_name"] = "grok-3-mini"
        return result

    if folder_name == "rational-all":
        result["rational_share"] = True
        result["rational_search"] = True
        result["rational_price"] = True
        return result

    if "_" in folder_name:
        result["model_name"] = folder_name.split("_")[-1]
    return result

def get_config_label(config: Dict) -> str:
    """Generate display label for configuration"""
    labels = []
    if any([config["rational_share"], config["rational_search"], config["rational_price"]]):
        rational_parts = []
        if config["rational_share"]: rational_parts.append("Share")
        if config["rational_search"]: rational_parts.append("Search")
        if config["rational_price"]: rational_parts.append("Price")
        labels.append(f"Rational: {', '.join(rational_parts)}")
    else:
        labels.append("LLM Simulation")
    if config["enable_cot"]:
        labels.append("CoT Enabled")
    if config["memory"]:
        labels.append(f"Memory (Length={config['memory_truncate']})")
    if config["share_memory"]:
        labels.append(f"Share Memory (Length={config['share_memory_truncate']})")
    if config["model_name"]:
        labels.append(f"Model: {config['model_name']}")
    return ", ".join(labels)

def create_safe_dir_name(config: Dict) -> str:
    """Create safe directory name for configuration"""
    parts = []
    if config["rational_share"] or config["rational_search"] or config["rational_price"]:
        rational_parts = []
        if config["rational_share"]: rational_parts.append("share")
        if config["rational_search"]: rational_parts.append("search")
        if config["rational_price"]: rational_parts.append("price")
        parts.append("rational_" + "_".join(rational_parts))
    else:
        parts.append("llm")
    if config["enable_cot"]:
        parts.append("cot")
    if config["memory"]:
        parts.append(f"memory_{config['memory_truncate']}")
    if config["share_memory"]:
        parts.append(f"sharememory_{config['share_memory_truncate']}")
    if config["model_name"]:
        parts.append(config["model_name"].replace("-", "_"))
    return "_".join(parts)

def find_data_files(base_dir: str) -> List[Tuple[Dict, str, int]]:
    """Find all experiment data files and parse configurations"""
    experiment_data = []
    print(f"Scanning directory: {base_dir}")
    if not os.path.exists(base_dir):
        print(f"❌ Error: Directory does not exist - {base_dir}")
        return []
    for folder in os.listdir(base_dir):
        folder_path = os.path.join(base_dir, folder)
        if not os.path.isdir(folder_path):
            continue
        config = parse_experiment_folder(folder)
        csv_files = glob.glob(os.path.join(folder_path, "experiment_*_firms_*_data.csv"))
        if not csv_files:
            continue
        for file_path in csv_files:
            match = re.search(r'firms_(\d+)_', os.path.basename(file_path))
            if match:
                firm_num = int(match.group(1))
                experiment_data.append((config, file_path, firm_num))
            else:
                print(f"  ⚠️ Cannot parse number of firms from filename: {os.path.basename(file_path)}")
    return experiment_data

def load_and_process_data(file_path: str, firm_num: int) -> pd.DataFrame:
    """Load and process single experiment data file"""
    try:
        lines = []
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.startswith('#') or line.strip() == '' or line.startswith('Experiment Configuration:') or line.startswith('Firm Count:'):
                    continue
                lines.append(line)
        if len(lines) < 2:
            return pd.DataFrame()
        try:
            df = pd.read_csv(file_path, comment='#', skipinitialspace=True)
        except pd.errors.ParserError:
            df = pd.read_csv(file_path, comment='#', delimiter='\t', skipinitialspace=True)
        column_mapping = {
            "Round": "round",
            "share_ratio": "share_ratio",
            "consumer_surplus": "consumer_surplus",
            "firm_surplus": "firm_surplus",
            "avg_search_cost": "avg_search_cost",
            "avg_price": "avg_price"
        }
        df = df.rename(columns={k: v for k, v in column_mapping.items() if k in df.columns})
        df['firm_num'] = firm_num
        if "rational-all" in os.path.basename(file_path).lower() and 'firm_surplus' in df.columns:
            df['firm_surplus'] = df['firm_surplus'] * 20
        if 'round' in df.columns:
            df['round'] = pd.to_numeric(df['round'], errors='coerce')
            df = df.dropna(subset=['round'])
            df['round'] = df['round'].astype(int)
            df = df.sort_values(by=['round']).reset_index(drop=True)
        return df
    except Exception as e:
        print(f"❌ Failed to load file {file_path}: {str(e)}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()

def load_and_process_data_(file_path: str, firm_num: int) -> pd.DataFrame:
    """Load and process single experiment data file"""
    try:
        import io
        import csv
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = []
            for line in f:
                if line.startswith('#') or line.strip() == '' or line.startswith('Experiment Configuration:') or line.startswith('Firm Count:'):
                    continue
                lines.append(line)
        if len(lines) < 2:
            return pd.DataFrame()
        csv_content = ''.join(lines)
        try:
            df = pd.read_csv(io.StringIO(csv_content), skipinitialspace=True)
        except pd.errors.ParserError:
            try:
                df = pd.read_csv(io.StringIO(csv_content), delimiter='\t', skipinitialspace=True)
            except pd.errors.ParserError:
                # More compatible fallback: use csv.Sniffer to detect delimiter
                sample = ''.join(lines[:5])  # Sample first few lines
                dialect = csv.Sniffer().sniff(sample)
                df = pd.read_csv(io.StringIO(csv_content), dialect=dialect, skipinitialspace=True)
        column_mapping = {
            "Round": "round",
            "share_ratio": "share_ratio",
            "consumer_surplus": "consumer_surplus",
            "firm_surplus": "firm_surplus",
            "avg_search_cost": "avg_search_cost",
            "avg_price": "avg_price"
        }
        df = df.rename(columns={k: v for k, v in column_mapping.items() if k in df.columns})
        df['firm_num'] = firm_num
        if "rational-all" in os.path.basename(file_path).lower() and 'firm_surplus' in df.columns:
            df['firm_surplus'] = df['firm_surplus'] * 20
        if 'round' in df.columns:
            df['round'] = pd.to_numeric(df['round'], errors='coerce')
            df = df.dropna(subset=['round'])
            df['round'] = df['round'].astype(int)
            df = df.sort_values(by=['round']).reset_index(drop=True)
        return df
    except Exception as e:
        print(f"❌ Failed to load file {file_path}: {str(e)}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()

def has_complete_data(data: pd.DataFrame) -> bool:
    """Check if data is complete"""
    metric_columns = [col for col, _ in METRICS]
    existing_metric_columns = [col for col in metric_columns if col in data.columns]
    if data[existing_metric_columns].isnull().values.any():
        print("  ⚠️ Data has missing values - skip plotting")
        return False
    return True

def calculate_confidence_interval(data: pd.Series, confidence=0.95) -> Tuple[float, float, float]:
    """Calculate confidence interval"""
    n = len(data)
    if n == 0:
        return np.nan, np.nan, np.nan
    mean = np.mean(data)
    if n < 2:
        return mean, mean, mean
    std = np.std(data, ddof=1)
    std_err = std / np.sqrt(n)
    t_val = stats.t.ppf((1 + confidence) / 2, n - 1)
    ci = t_val * std_err
    return mean, mean - ci, mean + ci

def plot_metric_with_ci(ax, data: pd.DataFrame, metric_col: str, x_col: str = 'firm_num', label: str = "Experiment",
                        line_style: str = '-', marker: str = 'o', color: str = None, confidence_level: float = 0.95,
                        show_data_points: bool = False) -> None:
    """Plot metric change chart with confidence interval"""
    grouped = data.groupby(x_col)[metric_col]
    firm_nums = sorted(data[x_col].unique())
    means, ci_lows, ci_highs = [], [], []
    for firm_num in firm_nums:
        if firm_num in grouped.groups:
            group_data = grouped.get_group(firm_num)
            mean, ci_low, ci_high = calculate_confidence_interval(group_data, confidence_level)
            means.append(mean)
            ci_lows.append(ci_low)
            ci_highs.append(ci_high)
        else:
            means.append(np.nan)
            ci_lows.append(np.nan)
            ci_highs.append(np.nan)
    valid_indices = ~np.isnan(means)
    valid_firm_nums = np.array(firm_nums)[valid_indices]
    valid_means = np.array(means)[valid_indices]
    valid_ci_lows = np.array(ci_lows)[valid_indices]
    valid_ci_highs = np.array(ci_highs)[valid_indices]
    if len(valid_firm_nums) == 0:
        return
    ax.plot(valid_firm_nums, valid_means, label=label, linestyle=line_style, marker=marker, linewidth=3, markersize=10, color=color)
    ax.fill_between(valid_firm_nums, valid_ci_lows, valid_ci_highs, alpha=0.3, color=color)
    if show_data_points:
        ax.scatter(data[x_col], data[metric_col], alpha=0.3, color=color)
    # Set x-axis tick interval to 1
    ax.set_xticks(np.arange(min(valid_firm_nums, default=0), max(valid_firm_nums, default=0) + 1, 1))
    # Increase and bolden x and y axis tick labels
    ax.tick_params(axis='both', which='major', labelsize=14, labelcolor='black', width=1.5)

def plot_mae_vs_memory(output_dir: str):
    """Plot memory length vs MAE based on mae_summary.csv (Modified version: only individual memory, including memory=0, add horizontal dashed line)"""
    mae_file = os.path.join(output_dir, "mae_summary.csv")
    if not os.path.exists(mae_file):
        print(f"⚠️ mae_summary.csv file does not exist: {mae_file} - Skip plotting memory length vs MAE relationship")
        return

    df = pd.read_csv(mae_file)
    models_to_plot = ["deepseek-v3", "gpt-4o-mini", "grok-3-mini", "gpt-3.5-turbo", "gemini-1.5-flash-8b", "gemini-2.0-flash", "gpt-4.1-mini"]
    memory_data = []

    for _, row in df.iterrows():
        config_label = row["Configuration"]
        config_dict = parse_config_label(config_label)

        is_llm_simulation = config_dict["rational"] is None
        not_cot = not config_dict["cot"]

        model_name = config_dict["model"]
        if model_name not in models_to_plot:
            continue

        if config_dict["memory"] and not config_dict["share_memory"]:
            memory_length = config_dict["memory_truncate"]
            memory_type = "Individual Memory"

            base_config_key = (
                config_dict["model"],
                config_dict["rational"],
                config_dict["cot"],
                config_dict["share_memory"]
            )

            for metric, _ in METRICS:
                if metric in row and not pd.isna(row[metric]):
                    memory_data.append({
                        "Model": model_name,
                        "Memory Length": memory_length,
                        "Metric": metric,
                        "MAE": row[metric],
                        "Memory Type": memory_type,
                        "Base Config Key": base_config_key
                    })

    if not memory_data:
        print("⚠️ No required memory distillation experiment data found in mae_summary.csv")
        return

    memory_df = pd.DataFrame(memory_data)
    memory_dir = os.path.join(output_dir, "memory_analysis")
    os.makedirs(memory_dir, exist_ok=True)

    for metric, title in METRICS:
        plt.figure(figsize=(12, 7))
        ax = plt.gca()

        individual_memory_data = memory_df[(memory_df["Metric"] == metric) &
                                           (memory_df["Memory Type"] == "Individual Memory")].copy()

        if individual_memory_data.empty:
            print(f"⚠️ Metric '{metric}' has no data points available for memory length chart (Individual Memory).")
            plt.close()
            continue

        valid_base_configs = set()
        for base_config_key, group_df in individual_memory_data.groupby("Base Config Key"):
            has_zero = (group_df["Memory Length"] == 0).any()
            has_positive = (group_df["Memory Length"] > 0).any()
            if has_zero and has_positive:
                valid_base_configs.add(base_config_key)

        filtered_for_plot = individual_memory_data[
            individual_memory_data["Base Config Key"].isin(valid_base_configs)
        ]

        if filtered_for_plot.empty:
            print(f"⚠️ Metric '{metric}' has no data points available for plotting memory length chart (meeting the condition 'configurations identical except for memory length').")
            plt.close()
            continue

        all_memory_lengths = sorted(filtered_for_plot["Memory Length"].unique())
        memory_0_maes = filtered_for_plot[filtered_for_plot["Memory Length"] == 0].set_index("Model")["MAE"].to_dict()

        for model in models_to_plot:
            model_specific_data = filtered_for_plot[filtered_for_plot["Model"] == model].sort_values("Memory Length")
            if not model_specific_data.empty:
                line, = ax.plot(model_specific_data["Memory Length"], model_specific_data["MAE"], marker='o', linestyle='-',
                                label=f"{model} (Individual Memory)", linewidth=4, alpha=0.8)
                if model in memory_0_maes:
                    ax.axhline(y=memory_0_maes[model], color='red', linestyle='--',
                               linewidth=4, alpha=0.7, label=f'_{model}_nolegend_')
                    ax.text(ax.get_xlim()[0] + 0.01 * (ax.get_xlim()[1] - ax.get_xlim()[0]), memory_0_maes[model],
                            f' {memory_0_maes[model]:.3f}', color=line.get_color(), va='center', ha='left', fontsize=9)

        # 删除xlabel和ylabel
        # plt.xlabel("Memory Length", fontsize=14)  # 已注释
        # plt.ylabel("MAE", fontsize=14)  # 已注释
        plt.grid(True, alpha=0.3)
        plt.tick_params(axis='both', which='major', labelsize=12)

        handles, labels = ax.get_legend_handles_labels()
        unique_handles = []
        unique_labels = []
        for handle, label in zip(handles, labels):
            if label.startswith('_'):
                continue
            if label not in unique_labels:
                unique_handles.append(handle)
                unique_labels.append(label)

        unique_handles.append(plt.Line2D([0], [0], color='red', linestyle='--', linewidth=1))
        unique_labels.append("Memory=0 Baseline")

        plt.xticks(all_memory_lengths, [f"{int(x)}" for x in all_memory_lengths])
        if len(all_memory_lengths) > 1:
            x_min, x_max = min(all_memory_lengths), max(all_memory_lengths)
            if x_min == x_max:
                plt.xlim(x_min * 0.9, x_max * 1.1)
            else:
                plt.xlim(x_min - (x_max - x_min) * 0.1, x_max + (x_max - x_min) * 0.1)
        elif len(all_memory_lengths) == 1:
            x_val = all_memory_lengths[0]
            plt.xlim(x_val - 1, x_val + 1)

        plt.minorticks_off()
        save_path = os.path.join(memory_dir, f"figure3_{metric}_mae_vs_individual_memory.png")
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"📊 Chart saved: {os.path.basename(save_path)}")
    print(f"✅ Memory length vs MAE relationship chart (only individual memory, including memory=0) saved to: {memory_dir}")

def plot_memory_10_bar_chart(output_dir: str):
    """
    Plot a composite bar chart for Individual Memory and Share Memory with memory length 10,
    and add a baseline bar for memory=0.
    X-axis is MAE metrics, Y-axis is MAE values, three adjacent bars under each metric.
    Use blue and orange tones for colors, horizontal x-axis labels.
    """
    mae_file = os.path.join(output_dir, "mae_summary.csv")
    if not os.path.exists(mae_file):
        print(f"⚠️ mae_summary.csv file does not exist: {mae_file} - Skip plotting bar chart for memory length 10.")
        return

    df = pd.read_csv(mae_file)
    plot_data = []
    models_to_plot = ["deepseek-v3", "gpt-4o-mini", "grok-3-mini", "gpt-3.5-turbo", "gemini-1.5-flash-8b", "gemini-2.0-flash", "gpt-4.1-mini"]

    for _, row in df.iterrows():
        config_label = row["Configuration"]
        config_dict = parse_config_label(config_label)

        is_llm_simulation = config_dict["rational"] is None
        not_cot = not config_dict["cot"]
        model_name = config_dict["model"]

        if not is_llm_simulation or not not_cot or model_name not in models_to_plot:
            continue

        data_to_add = None
        if config_dict["memory"] and not config_dict["share_memory"] and config_dict["memory_truncate"] == 0:
            data_to_add = {"Memory Type": "Memory=0 Baseline"}
        elif config_dict["memory"] and not config_dict["share_memory"] and config_dict["memory_truncate"] == 10:
            data_to_add = {"Memory Type": "Individual Memory (Length=10)"}
        elif config_dict["share_memory"] and config_dict["share_memory_truncate"] == 10:
            data_to_add = {"Memory Type": "Share Memory (Length=10)"}

        if data_to_add:
            for metric, _ in METRICS:
                if metric in row and not pd.isna(row[metric]):
                    entry = {
                        "Model": model_name,
                        "Metric": metric,
                        "MAE": row[metric]
                    }
                    entry.update(data_to_add)
                    plot_data.append(entry)

    if not plot_data:
        print("⚠️ No memory data found for plotting (Length 0, 10 Individual, or 10 Share), skip plotting.")
        return

    plot_df = pd.DataFrame(plot_data)
    bar_chart_dir = os.path.join(output_dir, "memory_10_bar_charts")
    os.makedirs(bar_chart_dir, exist_ok=True)

    metric_labels_display = [
        'Share\nRatio', 'Average\nPrice', 'Average\nConsumer\nSurplus',
        'Average\nFirm\nSurplus', 'Average\nSearch\nCost'
    ]
    custom_palette = ["#aec7e8", "#1f77b4", "#ff7f0e"]
    hue_order = ["Memory=0 Baseline", "Individual Memory (Length=10)", "Share Memory (Length=10)"]

    for model in models_to_plot:
        model_df = plot_df[plot_df["Model"] == model]
        if model_df.empty:
            continue

        plt.figure(figsize=(14, 8))
        ax = sns.barplot(data=model_df, x="Metric", y="MAE", hue="Memory Type",
                         palette=custom_palette,
                         hue_order=hue_order,
                         errorbar=None,
                         order=[m[0] for m in METRICS],
                         legend=False)  # 删除legends
        ax.set_xlabel("")
        ax.set_ylabel("")

        # 删除xlabel和ylabel
        # plt.xlabel("Metric", fontsize=24)  # 已注释
        # plt.ylabel("Mean Absolute Error (MAE)", fontsize=24)  # 已注释

        plt.xticks(ticks=range(len(METRICS)), labels=metric_labels_display, rotation=0, ha='center', fontsize=18)
        plt.yticks(fontsize=18)
        plt.grid(axis='y', alpha=0.3)

        # 删除每个柱子上面的数字
        # for container in ax.containers:
        #     ax.bar_label(container, fmt='%.3f', fontsize=9, padding=3)

        plt.tight_layout()
        save_path = os.path.join(bar_chart_dir, f"figure4_{model}_memory_10_bar_chart_with_baseline.png")
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"📊 Chart saved: {os.path.basename(save_path)}")

    print(f"✅ Composite bar chart for memory length 10 (with baseline) saved to: {bar_chart_dir}")

def plot_baseline_line(ax, baseline_data: pd.DataFrame, metric_col: str, label: str = "Rational Baseline",
                       color: str = 'k', linestyle: str = '-', linewidth: float = 2, marker: str = 'o') -> None:
    """Plot theoretical baseline line"""
    if not baseline_data.empty and metric_col in baseline_data.columns:
        baseline_means = baseline_data.groupby('firm_num')[metric_col].mean().reset_index().sort_values('firm_num')
        ax.plot(baseline_means['firm_num'], baseline_means[metric_col], label=label, color=color, linestyle=linestyle,
                linewidth=linewidth, marker=marker)

def calculate_distance_to_baseline(exp_data: pd.DataFrame, baseline_data: pd.DataFrame) -> Dict[str, float]:
    """Calculate average distance from experiment data to theoretical baseline"""
    distances = {}
    if baseline_data.empty:
        return {}
    common_firm_nums = set(exp_data['firm_num']).intersection(set(baseline_data['firm_num']))
    for metric, _ in METRICS:
        if metric in exp_data.columns and metric in baseline_data.columns:
            metric_distances = []
            for firm_num in common_firm_nums:
                exp_mean = exp_data[exp_data['firm_num'] == firm_num][metric].mean()
                baseline_mean = baseline_data[baseline_data['firm_num'] == firm_num][metric].mean()
                if not pd.isna(exp_mean) and not pd.isna(baseline_mean):
                    metric_distances.append(abs(exp_mean - baseline_mean))
            if metric_distances:
                distances[metric] = np.mean(metric_distances)
    if 'consumer_surplus' in exp_data.columns and 'firm_surplus' in exp_data.columns and \
       'consumer_surplus' in baseline_data.columns and 'firm_surplus' in baseline_data.columns:
        sw_distances = []
        for firm_num in common_firm_nums:
            exp_cs = exp_data[exp_data['firm_num'] == firm_num]['consumer_surplus'].mean()
            exp_fs = exp_data[exp_data['firm_num'] == firm_num]['firm_surplus'].mean()
            exp_sw = 20 * exp_cs + firm_num * exp_fs
            baseline_cs = baseline_data[baseline_data['firm_num'] == firm_num]['consumer_surplus'].mean()
            baseline_fs = baseline_data[baseline_data['firm_num'] == firm_num]['firm_surplus'].mean()
            baseline_sw = 20 * baseline_cs + firm_num * baseline_fs
            if not pd.isna(exp_sw) and not pd.isna(baseline_sw):
                sw_distances.append(abs(exp_sw - baseline_sw))
        if sw_distances:
            distances['social_welfare'] = np.mean(sw_distances)
    if distances:
        distances['overall'] = np.mean([v for k, v in distances.items() if k != 'overall'])
    return distances

def save_config_results(config: Dict, data: pd.DataFrame, output_dir: str, baseline_data: pd.DataFrame,
                        baseline_config: Dict = None) -> Dict[str, float]:
    """Save all results for a single configuration"""
    config_dir = create_safe_dir_name(config)
    config_path = os.path.join(output_dir, config_dir)
    os.makedirs(config_path, exist_ok=True)
    config_label = get_config_label(config)
    with open(os.path.join(config_path, "config_info.txt"), 'w', encoding='utf-8') as f:
        f.write(f"Configuration Details:\n")
        f.write(f"  Original Folder: {config['original_folder']}\n")
        f.write(f"  Rational Share: {config['rational_share']}\n")
        f.write(f"  Rational Search: {config['rational_search']}\n")
        f.write(f"  Rational Price: {config['rational_price']}\n")
        f.write(f"  Enable CoT: {config['enable_cot']}\n")
        f.write(f"  Memory: {config['memory']}\n")
        f.write(f"  Memory Truncate: {config['memory_truncate']}\n")
        f.write(f"  Share Memory: {config['share_memory']}\n")
        f.write(f"  Share Memory Truncate: {config['share_memory_truncate']}\n")
        f.write(f"  Model Name: {config['model_name']}\n")
        f.write(f"  Display Label: {config_label}\n")
    print(f"\n📁 Saving configuration results to: {config_path}")
    print(f"  Configuration: {config_label}")
    data_file = os.path.join(config_path, "experiment_data.csv")
    data.to_csv(data_file, index=False)
    print(f"  Raw data saved: {os.path.basename(data_file)}")
    distance_results = {}
    if not baseline_data.empty:
        distance_results = calculate_distance_to_baseline(data, baseline_data)
        distance_file = os.path.join(config_path, "distance_to_baseline.txt")
        with open(distance_file, 'w', encoding='utf-8') as f:
            if not distance_results:
                f.write("Cannot calculate distance to theoretical baseline: no common firm_num or metric data")
            else:
                f.write("Average absolute distance to theoretical baseline (rational-all):\n\n")
                for metric, dist in distance_results.items():
                    if metric != 'overall':
                        f.write(f"{metric}: {dist:.6f}\n")
                if 'overall' in distance_results:
                    f.write(f"\nOverall average distance: {distance_results['overall']:.6f}")
        print(f"  Distance calculation results saved: {os.path.basename(distance_file)}")
        if 'overall' in distance_results:
            print(f"  Average distance to theoretical baseline: {distance_results['overall']:.4f}")
    else:
        print("  No available theoretical baseline data - skip distance calculation")
    if not has_complete_data(data):
        print("  ⚠️ Data incomplete - skip plotting")
        return distance_results
    for metric, title in METRICS:
        if metric not in data.columns:
            continue
        fig, ax = plt.subplots(figsize=(8, 6))
        plot_metric_with_ci(ax, data, metric, label="90% CI", color='#1f77b4', confidence_level=0.90)
        plot_metric_with_ci(ax, data, metric, label="68% CI", color='#ff7f0e', confidence_level=0.68)
        if not baseline_data.empty:
            plot_baseline_line(ax, baseline_data, metric, label="Rational Baseline", color='black', linestyle='--', linewidth=2)
        # 删除标题、xlabel、ylabel和legends
        # ax.set_title(config['model_name'], fontsize=24)  # 已注释
        # ax.set_xlabel('Number of Firms', fontsize=14)  # 已注释
        # ax.set_ylabel(title, fontsize=14)  # 已注释
        ax.grid(True, alpha=0.3)
        # ax.legend(loc='best', fontsize=10)  # 已注释
        # 添加 'figure1_' 前缀到文件名
        ci_file = os.path.join(config_path, f"figure1_confidence_interval_{metric}.png")
        plt.tight_layout()
        plt.savefig(ci_file, dpi=300, bbox_inches='tight')
        plt.close(fig)
        print(f"  Saved confidence interval plot for {title}: {os.path.basename(ci_file)}")
    return distance_results

def find_baseline_data(experiment_data: List[Tuple[Dict, pd.DataFrame]]) -> Tuple[pd.DataFrame, Dict]:
    """Find rational-all configuration as theoretical baseline"""
    baseline_data = pd.DataFrame()
    baseline_config = None
    target_features = {
        "rational_share": True,
        "rational_search": True,
        "rational_price": True,
        "enable_cot": False,
        "memory": False,
        "share_memory": False
    }
    for config, data in experiment_data:
        if all(config[k] == v for k, v in target_features.items()):
            baseline_data = data
            baseline_config = config
            print(f"\n✅ Found theoretical baseline configuration: {get_config_label(config)}")
            break
    if baseline_data.empty:
        print("\n⚠️ Warning: No data found for rational-all configuration. Theoretical baseline will not be used.")
    return baseline_data, baseline_config

def visualize_results(experiment_data: List[Tuple[Dict, pd.DataFrame]], baseline_data: pd.DataFrame,
                      baseline_config: Dict, output_dir: str = "analysis_results") -> Dict[str, Dict[str, float]]:
    """Visualize all experiment results"""
    os.makedirs(output_dir, exist_ok=True)
    print(f"\nSaving results to: {output_dir}")
    all_distances = {}
    for config, data in experiment_data:
        if data.empty:
            print(f"⚠️ Configuration {get_config_label(config)} has no valid data, skip")
            continue
        distances = save_config_results(config, data, output_dir, baseline_data, baseline_config)
        all_distances[get_config_label(config)] = distances
    return all_distances

def create_mae_summary_table(distance_results: Dict[str, Dict[str, float]], output_dir: str) -> None:
    """Create and save MAE summary table"""
    if not distance_results:
        print("⚠️ No available distance results - skip creating MAE summary table")
        return
    summary_data = []
    metric_names = [metric for metric, _ in METRICS] + ['social_welfare']
    for config_label, metrics in distance_results.items():
        row = {"Configuration": config_label}
        for metric in metric_names:
            row[metric] = metrics.get(metric, np.nan)
        row['overall'] = metrics.get('overall', np.nan)
        summary_data.append(row)
    df = pd.DataFrame(summary_data)
    columns = ["Configuration"] + metric_names + ["overall"]
    df = df[columns]
    mae_file = os.path.join(output_dir, "mae_summary.csv")
    df.to_csv(mae_file, index=False)
    print(f"\n📊 MAE summary table saved: {os.path.basename(mae_file)}")

def extract_config(config_label: str) -> str:
    """Extract concise description from configuration label"""
    parts = config_label.split(", ")
    config_parts = []
    is_llm_simulation = True
    for part in parts:
        if part.startswith("Rational:"):
            config_parts.append(f"Rational {part.split(': ')[1]}")
            is_llm_simulation = False
        elif part.startswith("Memory (Length="):
            config_parts.append(f"Mem {part.split('=')[1].strip(')')}")
        elif part.startswith("Share Memory (Length="):
            config_parts.append(f"Share Mem {part.split('=')[1].strip(')')}")
        elif part == "CoT Enabled":
            config_parts.append("CoT")
    if is_llm_simulation and not config_parts:
        return "Classical LLM"
    elif is_llm_simulation:
        return "LLM " + ", ".join(config_parts)
    else:
        return ", ".join(config_parts)

def calculate_rational_step_improvement(output_dir: str) -> pd.DataFrame:
    """Calculate improvement of rational decision steps"""
    mae_file = os.path.join(output_dir, "mae_summary.csv")
    if not os.path.exists(mae_file):
        print(f"⚠️ MAE summary file does not exist: {mae_file}")
        return None
    df = pd.read_csv(mae_file)
    pure_llm_configs = df[~df['Configuration'].str.contains("Rational:") &
                          ~df['Configuration'].str.contains("CoT Enabled") &
                          df['Configuration'].str.contains("Memory \(Length=0\)")].copy()
    improvement_data = []
    steps = ["Share", "Search", "Price"]
    for _, pure_row in pure_llm_configs.iterrows():
        base_config_label = pure_row["Configuration"]
        config_dict = parse_config_label(base_config_label)
        model = config_dict["model"]
        if config_dict["cot"] or config_dict["memory_truncate"] != 0:
            continue
        for step in steps:
            rational_config_dict = config_dict.copy()
            rational_config_dict["rational"] = step
            rational_config_label = generate_config_label(rational_config_dict)
            rational_row = df[df["Configuration"] == rational_config_label]
            if not rational_row.empty:
                rational_row = rational_row.iloc[0]
                improvement = {
                    "Base Configuration": base_config_label,
                    "Rational Step": step,
                    "Model": model,
                    "Config (simplified)": extract_config(base_config_label)
                }
                for metric, _ in METRICS:
                    base_mae = pure_row[metric]
                    rational_mae = rational_row[metric]
                    improvement[metric] = base_mae - rational_mae if not pd.isna(base_mae) and not pd.isna(rational_mae) else np.nan
                improvement_data.append(improvement)
    if not improvement_data:
        print("⚠️ No matching rational step configurations found")
        return None
    improvement_df = pd.DataFrame(improvement_data)
    output_file = os.path.join(output_dir, "rational_step_improvement.csv")
    improvement_df.to_csv(output_file, index=False)
    print(f"✅ Rational step improvement results saved: {output_file}")
    return improvement_df

def plot_rational_step_improvement(output_dir: str):
    """Plot rational decision step improvement charts"""
    input_file = os.path.join(output_dir, "rational_step_improvement.csv")
    if not os.path.exists(input_file):
        print(f"⚠️ Rational step improvement file does not exist: {input_file}")
        return
    df = pd.read_csv(input_file)
    plots_dir = os.path.join(output_dir, "rational_step_improvement_plots")
    os.makedirs(plots_dir, exist_ok=True)
    for metric, title in METRICS:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
        fig.suptitle(f"Rational Step MAE Improvement - {title}", fontsize=18)
        for i, step in enumerate(["Share", "Search", "Price"]):
            ax = axes[i]
            step_df = df[df["Rational Step"] == step]
            models_order = sorted(step_df["Model"].unique())
            avg_improvements = step_df.groupby("Model")[metric].mean().reindex(models_order)
            bars = ax.bar(avg_improvements.index, avg_improvements.values,
                          color=['darkorange' if val >= 0 else 'steelblue' for val in avg_improvements.values],
                          edgecolor='black')
            for bar in bars:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2, height, f'{height:.2f}',
                        ha='center', va='bottom' if height >= 0 else 'top', fontsize=10)
            ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
            ax.set_title(f"Rational Decision: {step}", fontsize=14)
            ax.set_xlabel("Model", fontsize=12)
            if i == 0:
                ax.set_ylabel("MAE Reduction (Base - Rational)", fontsize=12)
            plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=10)
            ax.tick_params(axis='y', labelsize=10)
        plt.tight_layout(rect=[0, 0, 1, 0.93])
        plot_file = os.path.join(plots_dir, f"rational_step_improvement_{metric}.png")
        plt.savefig(plot_file, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"📊 Chart saved: {plot_file}")

def parse_config_label(config_label: str) -> Dict:
    """Parse configuration label"""
    parts = config_label.split(", ")
    config_dict = {
        "rational": None,
        "cot": False,
        "memory": False,
        "memory_truncate": 0,
        "share_memory": False,
        "share_memory_truncate": 0,
        "model": None
    }
    for part in parts:
        if part.startswith("Rational:"):
            config_dict["rational"] = part.split(": ")[1]
        elif part == "LLM Simulation":
            config_dict["rational"] = None
        elif part == "CoT Enabled":
            config_dict["cot"] = True
        elif part.startswith("Memory (Length="):
            config_dict["memory"] = True
            config_dict["memory_truncate"] = int(re.search(r"Length=(\d+)", part).group(1))
        elif part.startswith("Share Memory (Length="):
            config_dict["share_memory"] = True
            config_dict["share_memory_truncate"] = int(re.search(r"Length=(\d+)", part).group(1))
        elif part.startswith("Model:"):
            config_dict["model"] = part.split(": ")[1]
    return config_dict

def generate_config_label(config_dict: Dict) -> str:
    """Generate configuration label"""
    labels = []
    if config_dict["rational"]:
        labels.append(f"Rational: {config_dict['rational']}")
    else:
        labels.append("LLM Simulation")
    if config_dict["cot"]:
        labels.append("CoT Enabled")
    if config_dict["memory"]:
        labels.append(f"Memory (Length={config_dict['memory_truncate']})")
    if config_dict["share_memory"]:
        labels.append(f"Share Memory (Length={config_dict['share_memory_truncate']})")
    labels.append(f"Model: {config_dict['model']}")
    return ", ".join(labels)

def calculate_cot_improvement(output_dir: str) -> pd.DataFrame:
    """Calculate CoT improvement"""
    mae_file = os.path.join(output_dir, "mae_summary.csv")
    if not os.path.exists(mae_file):
        print(f"⚠️ MAE summary file does not exist: {mae_file}")
        return None
    df = pd.read_csv(mae_file)
    base_llm_configs = df[~df['Configuration'].str.contains("CoT Enabled") &
                          ~df['Configuration'].str.contains("Rational:") &
                          df['Configuration'].str.contains("Memory \(Length=0\)")].copy()
    improvement_data = []
    for _, base_row in base_llm_configs.iterrows():
        base_config_label = base_row["Configuration"]
        config_dict = parse_config_label(base_config_label)
        model = config_dict["model"]
        if config_dict["cot"] or config_dict["rational"] or config_dict["memory_truncate"] != 0:
            continue
        cot_config_dict = config_dict.copy()
        cot_config_dict["cot"] = True
        cot_config_label = generate_config_label(cot_config_dict)
        cot_row = df[df["Configuration"] == cot_config_label]
        if not cot_row.empty:
            cot_row = cot_row.iloc[0]
            improvement = {
                "Base Configuration": base_config_label,
                "Model": model,
                "Config (simplified)": extract_config(base_config_label)
            }
            for metric, _ in METRICS:
                base_mae = base_row[metric]
                cot_mae = cot_row[metric]
                improvement[metric] = base_mae - cot_mae if not pd.isna(base_mae) and not pd.isna(cot_mae) else np.nan
            improvement_data.append(improvement)
    if not improvement_data:
        print("⚠️ No matching CoT configurations found")
        return None
    improvement_df = pd.DataFrame(improvement_data)
    output_file = os.path.join(output_dir, "cot_improvement.csv")
    improvement_df.to_csv(output_file, index=False)
    print(f"✅ CoT improvement results saved: {output_file}")
    return improvement_df

def plot_cot_improvement(output_dir: str, plot_type: str = 'bar'):
    """Plot CoT improvement bar chart"""
    input_file = os.path.join(output_dir, "cot_improvement.csv")
    if not os.path.exists(input_file):
        print(f"⚠️ CoT improvement file does not exist: {input_file}")
        return
    df = pd.read_csv(input_file)
    plots_dir = os.path.join(output_dir, "cot_improvement_plots")
    os.makedirs(plots_dir, exist_ok=True)
    if plot_type != 'bar':
        print(f"⚠️ Current function only supports 'bar' type plotting, ignoring '{plot_type}' type.")
    all_models = ['grok-3-mini', 'deepseek-v3', 'gpt-4.1-mini', 'gemini-2.0-flash', 'gemini-1.5-flash-8b', 'gpt-3.5-turbo']
    unique_models_in_data = df['Model'].unique()
    custom_order = [m for m in all_models if m in unique_models_in_data]
    custom_order_for_plot = custom_order[::-1]
    for metric, title in METRICS:
        fig, ax = plt.subplots(figsize=(10, 6))
        improvements = df.groupby("Model")[metric].mean().reindex(custom_order_for_plot).dropna()
        if improvements.empty:
            print(f"⚠️ Metric '{metric}' has no CoT improvement data available for plotting.")
            plt.close(fig)
            continue
        models_to_plot = improvements.index.tolist()
        values_to_plot = improvements.values
        values_to_plot = np.sign(values_to_plot) * (np.log1p(np.abs(values_to_plot)))
        colors = ['darkorange' if val >= 0 else 'steelblue' for val in values_to_plot]
        bars = ax.barh(models_to_plot, values_to_plot, color=colors, edgecolor='black')
        # 删除每个柱子上的数字
        # for bar in bars:
        #     width = bar.get_width()
        #     y_pos = bar.get_y() + bar.get_height() / 2
        #     ha = 'left' if width >= 0 else 'right'
        #     x_offset = 0.01 * (ax.get_xlim()[1] - ax.get_xlim()[0])
        #     ax.text(width + (x_offset if width >= 0 else -x_offset), y_pos, f'{width:.2f}', ha=ha, va='center', fontsize=10)
        ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
        # 删除xlabel和标题
        # ax.set_xlabel("MAE Score Reduction (Base - CoT)", fontsize=12)  # 已注释
        ax.set_ylabel("")
        # ax.set_title(f"CoT Improvement for {title}", fontsize=14)  # 已注释
        ax.tick_params(axis='y', labelsize=12)
        if metric == 'share_ratio':
            ax.set_yticklabels(models_to_plot, fontsize=27)
        else:
            ax.set_yticklabels([])
        ax.grid(True, axis='x', alpha=0.3)
        plt.tight_layout()
        plot_file = os.path.join(plots_dir, f"figure2_cot_improvement_bar_{metric}.png")
        plt.savefig(plot_file, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"📊 Transposed bar chart saved: {plot_file}")

def calculate_social_welfare(experiment_data: List[Tuple[Dict, pd.DataFrame]], output_dir: str) -> None:
    """Calculate total social welfare"""
    welfare_data = []
    for config, data in experiment_data:
        if data.empty:
            continue
        config_label = get_config_label(config)
        grouped = data.groupby('firm_num')
        for firm_num, group in grouped:
            if 'consumer_surplus' in group.columns and 'firm_surplus' in group.columns:
                avg_consumer_surplus = group['consumer_surplus'].mean()
                avg_firm_surplus = group['firm_surplus'].mean()
                if not np.isnan(avg_consumer_surplus) and not np.isnan(avg_firm_surplus):
                    social_welfare = 20 * avg_consumer_surplus + firm_num * avg_firm_surplus
                    welfare_data.append({
                        "Configuration": config_label,
                        "Firm Number": firm_num,
                        "Social Welfare": social_welfare
                    })
    if not welfare_data:
        print("❌ No available social welfare data to save")
        return
    welfare_df = pd.DataFrame(welfare_data)
    os.makedirs(output_dir, exist_ok=True)
    welfare_file = os.path.join(output_dir, "social_welfare.csv")
    welfare_df.to_csv(welfare_file, index=False)
    print(f"\n📊 Social welfare table saved: {welfare_file}")

# Main execution flow
input_dir = "final_results_old"
output_dir = "analysis_results"
num_repeats = 10

print("="*60)
print("Starting experiment results analysis")
print("="*60)

data_files = find_data_files(input_dir)
experiment_data = []
config_data_map = {}
print(f"\nFound {len(data_files)} experiment data files")
for config, file_path, firm_num in data_files:
    data = load_and_process_data(file_path, firm_num)
    if not data.empty:
        config_key = (config["rational_share"], config["rational_search"], config["rational_price"],
                      config["enable_cot"], config["memory"], config["memory_truncate"],
                      config["share_memory"], config["share_memory_truncate"], config["model_name"])
        if config_key not in config_data_map:
            config_data_map[config_key] = {"config": config, "data": []}
        config_data_map[config_key]["data"].append(data)
final_experiment_data = []
for config_key, config_info in config_data_map.items():
    config = config_info["config"]
    combined_data = pd.concat(config_info["data"], ignore_index=True)
    firm_num_counts = combined_data.groupby('firm_num').size()
    valid_firm_nums = firm_num_counts[firm_num_counts == num_repeats].index
    if len(valid_firm_nums) > 0:
        filtered_data = combined_data[combined_data['firm_num'].isin(valid_firm_nums)].copy()
        filtered_data['original_folder'] = config['original_folder']
        final_experiment_data.append((config, filtered_data))
        print(f"✅ Configuration '{get_config_label(config)}' found {len(valid_firm_nums)} complete replicate data for firm numbers.")
    else:
        print(f"❌ Configuration '{get_config_label(config)}' found no complete replicate data for any firm number ({num_repeats} replicates). Skip this configuration.")

baseline_data, baseline_config = find_baseline_data(final_experiment_data)
print("\nVisualizing results and saving to individual configuration folders...")
distance_results = visualize_results(final_experiment_data, baseline_data, baseline_config, output_dir)
if distance_results:
    summary_file = os.path.join(output_dir, "distance_summary.csv")
    summary_data = []
    for config_label, metrics in distance_results.items():
        if metrics:
            for metric, dist in metrics.items():
                summary_data.append({"Configuration": config_label, "Metric": metric, "Distance": dist})
    if summary_data:
        summary_df = pd.DataFrame(summary_data)
        summary_df.to_csv(summary_file, index=False)
        print(f"\n📊 Distance summary results saved: {os.path.basename(summary_file)}")
        create_mae_summary_table(distance_results, output_dir)
        improvement_df = calculate_rational_step_improvement(output_dir)
        if improvement_df is not None:
            plot_rational_step_improvement(output_dir)
        cot_improvement_df = calculate_cot_improvement(output_dir)
        if cot_improvement_df is not None:
            plot_cot_improvement(output_dir, 'bar')
    else:
        print("\n⚠️ No distance data available for summary and subsequent analysis, skip generating MAE table, rational decision, and CoT improvement charts.")
else:
    print("\n⚠️ No distance data available for summary and subsequent analysis, skip generating MAE table, rational decision, and CoT improvement charts.")
plot_mae_vs_memory(output_dir)
plot_memory_10_bar_chart(output_dir)
calculate_social_welfare(final_experiment_data, output_dir)
print("\n✅ Analysis complete - results for each configuration saved in individual folders")
